# Stage 4A — Dimension-Level Portfolio Performance Analysis

This notebook calculates dimension-level portfolio performance only where Stage 3 eligibility and denominator gates permit it. It preserves source-native metric semantics, isolates strict-control primary results from sensitivity and ownership-pending evidence, and does not calculate a composite score or an overall portfolio winner.


## Environment Setup

Import the libraries used for deterministic input retrieval, integrity checks, tabular analysis, and output writing. Lock every analytical input to the verified Stage 3 publication commit.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
REPOSITORY_HEAD = "2648ddbe2b902f79826778a326e64ea9dc031e0f"
ANALYSIS_REFERENCE_DATE = "2026-08-21"

EXPECTED_MANIFEST_SHA256 = "c49038495822fc56abd171155f49a219db44a1605eee6cea146d272fda6f1d16"

REQUIRED_FILES = [
    "data/analytical/structural_portfolio_universe.csv",
    "data/analytical/competitive_observation_universe.csv",
    "data/analytical/group_category_period_denominators.csv",
    "data/analytical/longitudinal_series_eligibility.csv",
    "data/analytical/persistence_baseline_candidates.csv",
    "metadata/stage3_metric_eligibility_rules.csv",
    "metadata/stage3_preparation_validation.csv",
    "metadata/stage3_output_manifest.csv",
]

MANIFEST_TRACKED_INPUTS = [
    path for path in REQUIRED_FILES
    if path != "metadata/stage3_output_manifest.csv"
]

FOCAL_GROUPS = [
    "Wings Group",
    "Indofood",
    "Mayora",
    "Unilever Indonesia",
]

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE4A_OUTPUT_ROOT", "/content/stage4a_outputs")
)
ANALYTICAL_ROOT = OUTPUT_ROOT / "data" / "analytical"
METADATA_ROOT = OUTPUT_ROOT / "metadata"
ANALYTICAL_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)


## Locked Committed Input Retrieval

Retrieve the exact Stage 3 files from the locked commit. In Google Colab, the `GITHUB_TOKEN` secret is used only in the request authorization header and is never printed, stored in the repository, or embedded in a URL.


In [2]:
configured_root = os.environ.get("FMCG_STAGE4A_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in Google Colab or set FMCG_STAGE4A_INPUT_ROOT "
            "for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "The Colab Secret GITHUB_TOKEN is unavailable or access has not "
            "been granted to this notebook."
        )

    INPUT_ROOT = Path("/content/fmcg_stage4a_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in REQUIRED_FILES:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        api_url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/"
            f"{encoded_path}?ref={REPOSITORY_HEAD}"
        )
        request = urllib.request.Request(
            api_url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage4a-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} "
                f"with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing_inputs = [
    relative_path
    for relative_path in REQUIRED_FILES
    if not (INPUT_ROOT / relative_path).exists()
]
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Input mode: {input_mode}")
print(f"Locked repository head: {REPOSITORY_HEAD}")
print(f"Required files found: {len(REQUIRED_FILES)}/{len(REQUIRED_FILES)}")


Input mode: locked_github_commit
Locked repository head: 2648ddbe2b902f79826778a326e64ea9dc031e0f
Required files found: 8/8


## Stage 3 Manifest and Input Integrity Validation

Validate the committed Stage 3 manifest itself, then verify the SHA-256 and row count of every manifest-tracked analytical input before any performance calculation is allowed to run.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = INPUT_ROOT / "metadata/stage3_output_manifest.csv"
actual_manifest_sha256 = sha256_file(manifest_path)

if actual_manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise ValueError(
        "Stage 3 output manifest checksum differs from the locked expected value."
    )

stage3_manifest = pd.read_csv(
    manifest_path,
    dtype=str,
    keep_default_na=False,
)

manifest_paths = set(stage3_manifest["file_path"])
expected_manifest_paths = set(MANIFEST_TRACKED_INPUTS)

if manifest_paths != expected_manifest_paths:
    missing = sorted(expected_manifest_paths - manifest_paths)
    extra = sorted(manifest_paths - expected_manifest_paths)
    raise ValueError(
        f"Stage 3 manifest path mismatch. Missing={missing}; extra={extra}"
    )

integrity_rows = []
for row in stage3_manifest.itertuples(index=False):
    relative_path = row.file_path
    path = INPUT_ROOT / relative_path
    actual_sha256 = sha256_file(path)
    actual_row_count = len(
        pd.read_csv(path, dtype=str, keep_default_na=False)
    )
    expected_row_count = int(row.row_count)

    integrity_rows.append(
        {
            "file_path": relative_path,
            "expected_row_count": expected_row_count,
            "actual_row_count": actual_row_count,
            "expected_sha256": row.sha256,
            "actual_sha256": actual_sha256,
            "row_count_status": (
                "passed" if actual_row_count == expected_row_count else "failed"
            ),
            "sha256_status": (
                "passed" if actual_sha256 == row.sha256 else "failed"
            ),
        }
    )

input_integrity = pd.DataFrame(integrity_rows)

if not (
    input_integrity["row_count_status"].eq("passed").all()
    and input_integrity["sha256_status"].eq("passed").all()
):
    raise ValueError("Stage 3 input integrity validation failed.")

print(f"Stage 3 manifest SHA-256: passed")
print(
    input_integrity[
        ["file_path", "actual_row_count", "row_count_status", "sha256_status"]
    ].to_string(index=False)
)


Stage 3 manifest SHA-256: passed
                                             file_path  actual_row_count row_count_status sha256_status
     data/analytical/structural_portfolio_universe.csv               157           passed        passed
  data/analytical/competitive_observation_universe.csv               100           passed        passed
data/analytical/group_category_period_denominators.csv                82           passed        passed
   data/analytical/longitudinal_series_eligibility.csv                36           passed        passed
   data/analytical/persistence_baseline_candidates.csv                14           passed        passed
          metadata/stage3_metric_eligibility_rules.csv                11           passed        passed
            metadata/stage3_preparation_validation.csv                25           passed        passed


## Analytical Input Loading and Schema Gates

Load the locked Stage 3 analytical tables as strings, verify the required fields, and confirm that the Stage 3 gate contains no critical failure.


In [4]:
def read_table(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(
        INPUT_ROOT / relative_path,
        dtype=str,
        keep_default_na=False,
    )


structural = read_table(
    "data/analytical/structural_portfolio_universe.csv"
)
competitive = read_table(
    "data/analytical/competitive_observation_universe.csv"
)
denominators = read_table(
    "data/analytical/group_category_period_denominators.csv"
)
longitudinal = read_table(
    "data/analytical/longitudinal_series_eligibility.csv"
)
persistence = read_table(
    "data/analytical/persistence_baseline_candidates.csv"
)
metric_rules = read_table(
    "metadata/stage3_metric_eligibility_rules.csv"
)
stage3_validation = read_table(
    "metadata/stage3_preparation_validation.csv"
)

REQUIRED_COLUMNS = {
    "structural": {
        "group",
        "brand_family",
        "category_mapping_status",
        "current_at_reference_date",
        "strict_control_primary",
        "structural_brand_count_key",
        "structural_brand_breadth_eligible",
        "structural_category_breadth_eligible",
    },
    "competitive": {
        "observation_id",
        "universe_id",
        "source_family",
        "canonical_group",
        "attribution_scope",
        "source_brand_label",
        "canonical_brand_family",
        "canonical_brand_key",
        "source_category",
        "source_subcategory",
        "source_subcategory_id",
        "canonical_sector",
        "canonical_category",
        "canonical_subcategory",
        "reference_period",
        "geography",
        "methodology_cluster",
        "metric",
        "value",
        "unit",
        "observation_status",
        "ownership_registry_primary_match",
        "primary_analysis_eligible",
        "sensitivity_analysis_eligible",
        "analysis_scope",
        "value_available",
        "category_strength_candidate",
        "category_leadership_value_candidate",
        "consumer_reach_group_comparison_eligible",
        "source_label_series_key",
        "eligibility_exclusion_reason",
    },
    "denominators": {
        "canonical_group",
        "source_family",
        "source_category",
        "source_subcategory",
        "source_subcategory_id",
        "canonical_sector",
        "canonical_category",
        "canonical_subcategory",
        "reference_period",
        "methodology_cluster",
        "metric_scope",
        "attribution_scope",
        "eligible_target_count",
        "pending_ownership_target_count",
        "sensitivity_target_count",
        "observable_target_count",
        "observed_target_count",
        "explicit_not_available_target_count",
        "not_observed_target_count",
        "context_only_target_count",
        "standardized_observation_row_count",
        "group_denominator_status",
        "category_eligible_target_count",
        "category_pending_ownership_target_count",
        "category_observable_target_count",
        "category_observed_target_count",
        "category_explicit_not_available_target_count",
        "category_not_observed_target_count",
        "category_distinct_eligible_groups",
        "category_denominator_status",
    },
    "longitudinal": {
        "source_label_series_key",
        "canonical_group",
        "canonical_brand_family",
        "canonical_brand_key",
        "source_brand_label",
        "canonical_sector",
        "canonical_category",
        "canonical_subcategory",
        "source_subcategory_id",
        "metric",
        "unit",
        "geography",
        "methodology_cluster",
        "methodology_status",
        "ownership_support_status",
        "observed_period_count",
        "observed_periods",
        "maximum_consecutive_observed_periods",
        "consistency_eligibility",
        "momentum_eligibility",
        "methodology_caveat",
    },
    "persistence": {
        "source_family",
        "source_category",
        "source_subcategory",
        "source_subcategory_id",
        "canonical_sector",
        "canonical_category",
        "canonical_subcategory",
        "methodology_cluster",
        "baseline_status",
        "baseline_period",
        "baseline_incumbent_group",
        "baseline_incumbent_brand",
        "baseline_incumbent_observation_id",
        "baseline_tbi",
        "complete_follow_up_period_count",
        "complete_follow_up_periods",
        "persistence_analysis_eligibility",
        "eligibility_reason",
    },
    "metric_rules": {
        "dimension_id",
        "dimension",
        "current_eligibility",
        "prohibited_use",
        "eligibility_reason",
    },
    "stage3_validation": {
        "check_id",
        "result",
        "status",
        "critical_failure",
    },
}

TABLES = {
    "structural": structural,
    "competitive": competitive,
    "denominators": denominators,
    "longitudinal": longitudinal,
    "persistence": persistence,
    "metric_rules": metric_rules,
    "stage3_validation": stage3_validation,
}

for table_name, required_columns in REQUIRED_COLUMNS.items():
    missing_columns = sorted(
        required_columns - set(TABLES[table_name].columns)
    )
    if missing_columns:
        raise KeyError(
            f"{table_name} is missing required columns: {missing_columns}"
        )

if stage3_validation["critical_failure"].str.lower().eq("yes").any():
    raise RuntimeError("Stage 3 contains a critical validation failure.")

stage3_gate_rows = stage3_validation[
    stage3_validation["check_id"].eq("S3A025")
]
if len(stage3_gate_rows) != 1:
    raise ValueError("Stage 3 final gate row S3A025 is not unique.")
if stage3_gate_rows.iloc[0]["result"] != "PASS_WITH_CAVEAT":
    raise RuntimeError(
        "Stage 3 final gate is not the expected PASS_WITH_CAVEAT."
    )

print(
    pd.DataFrame(
        {
            "table": list(TABLES.keys()),
            "row_count": [len(TABLES[name]) for name in TABLES],
        }
    ).to_string(index=False)
)
print("Stage 3 gate: PASS_WITH_CAVEAT")


            table  row_count
       structural        157
      competitive        100
     denominators         82
     longitudinal         36
      persistence         14
     metric_rules         11
stage3_validation         25
Stage 3 gate: PASS_WITH_CAVEAT


## Structural Brand-Family Breadth

Count distinct canonical brand families only in the current strict-control primary ownership universe. Unresolved category mappings remain visible as a caveat and are not used to infer structural category breadth.


In [5]:
structural_primary_mask = (
    structural["current_at_reference_date"].eq("yes")
    & structural["strict_control_primary"].eq("yes")
    & structural["structural_brand_breadth_eligible"].eq("yes")
)

structural_primary = structural.loc[structural_primary_mask].copy()

structural_brand_breadth = (
    structural_primary.groupby("group", as_index=False)
    .agg(
        structural_brand_family_count=(
            "structural_brand_count_key",
            "nunique",
        ),
        current_strict_control_record_count=(
            "structural_brand_count_key",
            "size",
        ),
        unresolved_category_mapping_record_count=(
            "category_mapping_status",
            lambda series: int(series.eq("unresolved").sum()),
        ),
    )
    .rename(columns={"group": "canonical_group"})
)

structural_brand_breadth = (
    pd.DataFrame({"canonical_group": FOCAL_GROUPS})
    .merge(
        structural_brand_breadth,
        on="canonical_group",
        how="left",
        validate="one_to_one",
    )
)

count_columns = [
    "structural_brand_family_count",
    "current_strict_control_record_count",
    "unresolved_category_mapping_record_count",
]
for column in count_columns:
    structural_brand_breadth[column] = (
        structural_brand_breadth[column].fillna(0).astype(int)
    )

structural_brand_breadth["breadth_rank"] = (
    structural_brand_breadth["structural_brand_family_count"]
    .rank(method="min", ascending=False)
    .astype(int)
)

max_breadth = structural_brand_breadth[
    "structural_brand_family_count"
].max()
leader_count = int(
    structural_brand_breadth[
        "structural_brand_family_count"
    ].eq(max_breadth).sum()
)
structural_brand_breadth["leader_status"] = np.where(
    structural_brand_breadth[
        "structural_brand_family_count"
    ].eq(max_breadth),
    (
        "unique_structural_brand_breadth_leader"
        if leader_count == 1
        else "shared_structural_brand_breadth_leader"
    ),
    "not_leader",
)

structural_brand_breadth["structural_category_breadth_status"] = (
    "not_eligible_currently"
)
structural_brand_breadth["result_scope"] = (
    "current strict-control canonical brand-family breadth"
)
structural_brand_breadth["caveat"] = (
    "Structural category breadth is not calculated because comparable "
    "category mapping is incomplete, especially for Wings Group."
)

structural_brand_breadth = structural_brand_breadth.sort_values(
    ["breadth_rank", "canonical_group"]
).reset_index(drop=True)

print(structural_brand_breadth.to_string(index=False))


   canonical_group  structural_brand_family_count  current_strict_control_record_count  unresolved_category_mapping_record_count  breadth_rank                          leader_status structural_category_breadth_status                                          result_scope                                                                                                                       caveat
          Indofood                             38                                   39                                         0             1 unique_structural_brand_breadth_leader             not_eligible_currently current strict-control canonical brand-family breadth Structural category breadth is not calculated because comparable category mapping is incomplete, especially for Wings Group.
       Wings Group                             37                                   45                                        45             2                             not_leader             not_eligible_c

## Competitive Breadth with Explicit Denominators

Aggregate the Stage 3 eligible, observable, observed, unavailable, not-observed, pending-ownership, sensitivity, and context counts by group and source family. Top Brand counts remain denominator-explicit; limited Brand Footprint facts remain contextual and are not converted into a cross-group reach comparison.


In [6]:
DENOMINATOR_COUNT_COLUMNS = [
    "eligible_target_count",
    "pending_ownership_target_count",
    "sensitivity_target_count",
    "observable_target_count",
    "observed_target_count",
    "explicit_not_available_target_count",
    "not_observed_target_count",
    "context_only_target_count",
    "standardized_observation_row_count",
]

denominator_numeric = denominators.copy()
for column in DENOMINATOR_COUNT_COLUMNS:
    denominator_numeric[column] = pd.to_numeric(
        denominator_numeric[column].replace("", "0"),
        errors="raise",
    ).astype(int)

competitive_breadth = (
    denominator_numeric.groupby(
        ["canonical_group", "source_family"],
        as_index=False,
        dropna=False,
    )[DENOMINATOR_COUNT_COLUMNS]
    .sum()
)

competitive_breadth["expanded_primary_or_pending_or_sensitivity_count"] = (
    competitive_breadth["eligible_target_count"]
    + competitive_breadth["pending_ownership_target_count"]
    + competitive_breadth["sensitivity_target_count"]
)

competitive_breadth["quantitative_treatment"] = np.where(
    competitive_breadth["source_family"].eq("Top Brand"),
    "denominator_explicit_competitive_breadth_with_caveat",
    "context_only_limited_public_fact",
)
competitive_breadth["performance_ranking_status"] = (
    "not_ranked_due_selective_public_coverage"
)
competitive_breadth["caveat"] = np.where(
    competitive_breadth["source_family"].eq("Top Brand"),
    (
        "Counts describe the pre-specified observable competitive evidence "
        "universe. Omission and explicit unavailability are not zero-valued "
        "performance outcomes."
    ),
    (
        "Limited Brand Footprint public facts are retained as context only "
        "and do not provide a comparable CRP universe across focal groups."
    ),
)

competitive_breadth = competitive_breadth.sort_values(
    ["source_family", "canonical_group"]
).reset_index(drop=True)

print(competitive_breadth.to_string(index=False))


   canonical_group   source_family  eligible_target_count  pending_ownership_target_count  sensitivity_target_count  observable_target_count  observed_target_count  explicit_not_available_target_count  not_observed_target_count  context_only_target_count  standardized_observation_row_count  expanded_primary_or_pending_or_sensitivity_count                               quantitative_treatment               performance_ranking_status                                                                                                                                                     caveat
          Indofood Brand Footprint                      2                               0                         0                        2                      2                                    0                          0                          0                                   2                                                 2                     context_only_limited_public_fact not_ranked_due_s

## Top Brand Category Strength

Retain only observed, strict-control, primary-analysis TBI values. Compare values only within the same source subcategory, reference period, geography, unit, and methodology cluster. Focal observed positions are descriptive and are not full-market ranks.


In [7]:
CATEGORY_KEYS = [
    "source_family",
    "source_category",
    "source_subcategory",
    "source_subcategory_id",
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "reference_period",
    "methodology_cluster",
]

CATEGORY_DENOMINATOR_FIELDS = [
    "category_eligible_target_count",
    "category_pending_ownership_target_count",
    "category_observable_target_count",
    "category_observed_target_count",
    "category_explicit_not_available_target_count",
    "category_not_observed_target_count",
    "category_distinct_eligible_groups",
    "category_denominator_status",
]

top_brand_denominators = denominator_numeric[
    denominator_numeric["source_family"].eq("Top Brand")
].copy()

for field in CATEGORY_DENOMINATOR_FIELDS:
    inconsistent = (
        top_brand_denominators.groupby(CATEGORY_KEYS, dropna=False)[field]
        .nunique(dropna=False)
        .gt(1)
    )
    if inconsistent.any():
        bad_keys = inconsistent[inconsistent].index.tolist()[:5]
        raise ValueError(
            f"Inconsistent category denominator field {field}: {bad_keys}"
        )

category_denominator_meta = (
    top_brand_denominators[
        CATEGORY_KEYS + CATEGORY_DENOMINATOR_FIELDS
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

numeric_category_fields = [
    "category_eligible_target_count",
    "category_pending_ownership_target_count",
    "category_observable_target_count",
    "category_observed_target_count",
    "category_explicit_not_available_target_count",
    "category_not_observed_target_count",
    "category_distinct_eligible_groups",
]
for field in numeric_category_fields:
    category_denominator_meta[field] = pd.to_numeric(
        category_denominator_meta[field].replace("", "0"),
        errors="raise",
    ).astype(int)

strength_mask = (
    competitive["source_family"].eq("Top Brand")
    & competitive["primary_analysis_eligible"].eq("yes")
    & competitive["analysis_scope"].eq("primary_strict_control")
    & competitive["category_strength_candidate"].eq("yes")
    & competitive["value_available"].eq("yes")
    & competitive["metric"].eq("TBI")
    & competitive["unit"].eq("percent_index_points")
)

category_strength = competitive.loc[strength_mask].copy()
category_strength["tbi_value"] = pd.to_numeric(
    category_strength["value"],
    errors="raise",
)
category_strength["reference_period"] = pd.to_numeric(
    category_strength["reference_period"],
    errors="raise",
).astype(int)

category_denominator_meta["reference_period"] = pd.to_numeric(
    category_denominator_meta["reference_period"],
    errors="raise",
).astype(int)

category_strength = category_strength.merge(
    category_denominator_meta,
    on=CATEGORY_KEYS,
    how="left",
    validate="many_to_one",
)

if category_strength["category_denominator_status"].eq("").any():
    raise ValueError(
        "At least one category-strength observation lacks denominator metadata."
    )

category_strength["focal_observed_position"] = (
    category_strength.groupby(CATEGORY_KEYS, dropna=False)["tbi_value"]
    .rank(method="min", ascending=False)
    .astype(int)
)
category_strength["focal_observed_brand_count"] = (
    category_strength.groupby(CATEGORY_KEYS, dropna=False)[
        "observation_id"
    ].transform("count").astype(int)
)

leadership_gate = (
    category_strength["category_denominator_status"].eq("complete_observed")
    & category_strength["category_distinct_eligible_groups"].ge(2)
    & category_strength["category_pending_ownership_target_count"].eq(0)
    & category_strength["category_explicit_not_available_target_count"].eq(0)
    & category_strength["category_not_observed_target_count"].eq(0)
)
category_strength["leadership_eligible"] = np.where(
    leadership_gate, "yes", "no"
)
category_strength["comparison_scope"] = (
    "focal observed brands within one category-period-methodology cluster"
)
category_strength["position_semantics"] = (
    "Derived focal-observed position; not a full-market rank and not market share."
)

CATEGORY_STRENGTH_OUTPUT_COLUMNS = [
    "observation_id",
    "universe_id",
    "canonical_group",
    "source_brand_label",
    "canonical_brand_family",
    "canonical_brand_key",
    "source_category",
    "source_subcategory",
    "source_subcategory_id",
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "reference_period",
    "geography",
    "methodology_cluster",
    "metric",
    "unit",
    "tbi_value",
    "focal_observed_position",
    "focal_observed_brand_count",
    "category_eligible_target_count",
    "category_pending_ownership_target_count",
    "category_observable_target_count",
    "category_observed_target_count",
    "category_explicit_not_available_target_count",
    "category_not_observed_target_count",
    "category_distinct_eligible_groups",
    "category_denominator_status",
    "leadership_eligible",
    "comparison_scope",
    "position_semantics",
]

category_strength_results = (
    category_strength[CATEGORY_STRENGTH_OUTPUT_COLUMNS]
    .sort_values(
        [
            "source_subcategory_id",
            "methodology_cluster",
            "reference_period",
            "focal_observed_position",
            "canonical_group",
            "source_brand_label",
        ]
    )
    .reset_index(drop=True)
)

print(
    category_strength_results[
        [
            "canonical_group",
            "source_brand_label",
            "canonical_subcategory",
            "reference_period",
            "methodology_cluster",
            "tbi_value",
            "focal_observed_position",
            "category_denominator_status",
            "leadership_eligible",
        ]
    ].to_string(index=False)
)


   canonical_group source_brand_label       canonical_subcategory  reference_period             methodology_cluster  tbi_value  focal_observed_position             category_denominator_status leadership_eligible
       Wings Group   TOP White Coffee                White Coffee              2026    top_brand_current_documented       10.0                        1                       complete_observed                 yes
            Mayora  KOPIKO L.A. White                White Coffee              2026    top_brand_current_documented        6.0                        2                       complete_observed                 yes
       Wings Group   TOP White Coffee                White Coffee              2022 top_brand_historical_unverified        3.6                        1 observable_with_explicit_unavailability                  no
       Wings Group   TOP White Coffee                White Coffee              2023 top_brand_historical_unverified        7.5                        1 

## Conditional Focal-Group Category Leadership

Identify a leader only when the category-period denominator is complete, at least two focal groups are eligible, and no pending ownership, explicit unavailability, or not-observed target remains. The result is explicitly a focal-group leader, not a full-market category leader.


In [8]:
leadership_input = category_strength[
    category_strength["leadership_eligible"].eq("yes")
].copy()

if leadership_input.empty:
    category_leadership_results = pd.DataFrame(
        columns=CATEGORY_KEYS
        + [
            "focal_group_count",
            "leader_tbi",
            "leader_group_count",
            "leader_groups",
            "leader_brand_labels",
            "leader_brand_families",
            "group_best_snapshot",
            "leader_status",
            "result_scope",
            "category_denominator_status",
        ]
    )
else:
    leadership_input["group_best_tbi"] = (
        leadership_input.groupby(
            CATEGORY_KEYS + ["canonical_group"],
            dropna=False,
        )["tbi_value"].transform("max")
    )

    group_best_rows = leadership_input[
        np.isclose(
            leadership_input["tbi_value"],
            leadership_input["group_best_tbi"],
            rtol=0,
            atol=1e-12,
        )
    ].copy()

    def sorted_join(series):
        values = sorted({str(value) for value in series if str(value) != ""})
        return " | ".join(values)

    group_best = (
        group_best_rows.groupby(
            CATEGORY_KEYS + ["canonical_group"],
            as_index=False,
            dropna=False,
        )
        .agg(
            group_best_tbi=("group_best_tbi", "first"),
            group_best_brand_labels=("source_brand_label", sorted_join),
            group_best_brand_families=(
                "canonical_brand_family",
                sorted_join,
            ),
        )
    )

    leadership_rows = []
    for key_values, group_frame in group_best.groupby(
        CATEGORY_KEYS,
        dropna=False,
        sort=True,
    ):
        key_dict = dict(zip(CATEGORY_KEYS, key_values))
        leader_tbi = float(group_frame["group_best_tbi"].max())
        leaders = group_frame[
            np.isclose(
                group_frame["group_best_tbi"],
                leader_tbi,
                rtol=0,
                atol=1e-12,
            )
        ].copy()

        leader_groups = sorted(leaders["canonical_group"].unique())
        leader_brand_labels = "; ".join(
            f"{row.canonical_group}: {row.group_best_brand_labels}"
            for row in leaders.sort_values("canonical_group").itertuples()
        )
        leader_brand_families = "; ".join(
            f"{row.canonical_group}: {row.group_best_brand_families}"
            for row in leaders.sort_values("canonical_group").itertuples()
        )
        snapshot = "; ".join(
            (
                f"{row.canonical_group}: "
                f"{float(row.group_best_tbi):.6g} "
                f"({row.group_best_brand_labels})"
            )
            for row in group_frame.sort_values(
                ["group_best_tbi", "canonical_group"],
                ascending=[False, True],
            ).itertuples()
        )

        denominator_row = leadership_input
        for key, value in key_dict.items():
            denominator_row = denominator_row[
                denominator_row[key].eq(value)
            ]
        denominator_row = denominator_row.iloc[0]

        leadership_rows.append(
            {
                **key_dict,
                "focal_group_count": int(
                    group_frame["canonical_group"].nunique()
                ),
                "leader_tbi": leader_tbi,
                "leader_group_count": len(leader_groups),
                "leader_groups": " | ".join(leader_groups),
                "leader_brand_labels": leader_brand_labels,
                "leader_brand_families": leader_brand_families,
                "group_best_snapshot": snapshot,
                "leader_status": (
                    "unique_focal_group_leader"
                    if len(leader_groups) == 1
                    else "shared_focal_group_leader"
                ),
                "result_scope": (
                    "focal-group leader only; not full-market category leader"
                ),
                "category_denominator_status": (
                    denominator_row["category_denominator_status"]
                ),
            }
        )

    category_leadership_results = (
        pd.DataFrame(leadership_rows)
        .sort_values(
            [
                "source_subcategory_id",
                "methodology_cluster",
                "reference_period",
            ]
        )
        .reset_index(drop=True)
    )

print(category_leadership_results.to_string(index=False))


source_family     source_category                        source_subcategory source_subcategory_id canonical_sector        canonical_category       canonical_subcategory  reference_period             methodology_cluster  focal_group_count  leader_tbi  leader_group_count      leader_groups           leader_brand_labels        leader_brand_families                                                                       group_best_snapshot             leader_status                                             result_scope category_denominator_status
    Top Brand Makanan dan Minuman                              WHITE COFFEE                    10        Beverages                    Coffee                White Coffee              2026    top_brand_current_documented                  2        10.0                   1        Wings Group Wings Group: TOP White Coffee      Wings Group: TOP Coffee                         Wings Group: 10 (TOP White Coffee); Mayora: 6 (KOPIKO L.A. White) unique_focal_

## Longitudinal Consistency

Calculate observed stability only for Stage 3 consistency-eligible source-label series. Use the longest consecutive observed block within one category, geography, metric, unit, source label, and methodology cluster. Lower mean absolute period-to-period change indicates greater observed stability only within a directly comparable category-period block.


In [9]:
def longest_consecutive_block(periods):
    unique_periods = sorted({int(period) for period in periods})
    if not unique_periods:
        return []

    blocks = [[unique_periods[0]]]
    for period in unique_periods[1:]:
        if period == blocks[-1][-1] + 1:
            blocks[-1].append(period)
        else:
            blocks.append([period])

    blocks.sort(key=lambda block: (-len(block), block[0]))
    return blocks[0]


consistency_eligible = longitudinal[
    longitudinal["consistency_eligibility"].eq("eligible_with_caveat")
].copy()

consistency_rows = []
for series_row in consistency_eligible.itertuples(index=False):
    series_key = series_row.source_label_series_key

    observations = competitive[
        competitive["source_label_series_key"].eq(series_key)
        & competitive["primary_analysis_eligible"].eq("yes")
        & competitive["analysis_scope"].eq("primary_strict_control")
        & competitive["value_available"].eq("yes")
        & competitive["metric"].eq("TBI")
        & competitive["unit"].eq("percent_index_points")
    ].copy()

    observations["reference_period_int"] = pd.to_numeric(
        observations["reference_period"],
        errors="raise",
    ).astype(int)
    observations["tbi_value"] = pd.to_numeric(
        observations["value"],
        errors="raise",
    )

    if observations["reference_period_int"].duplicated().any():
        raise ValueError(
            f"Duplicate periods found in longitudinal series {series_key}."
        )

    analysis_periods = longest_consecutive_block(
        observations["reference_period_int"].tolist()
    )
    if len(analysis_periods) < 3:
        raise ValueError(
            f"Eligible consistency series lacks a 3-period block: {series_key}"
        )

    block = (
        observations[
            observations["reference_period_int"].isin(analysis_periods)
        ]
        .sort_values("reference_period_int")
        .copy()
    )
    values = block["tbi_value"].to_numpy(dtype=float)
    changes = np.diff(values)

    consistency_rows.append(
        {
            "source_label_series_key": series_key,
            "canonical_group": series_row.canonical_group,
            "canonical_brand_family": series_row.canonical_brand_family,
            "canonical_brand_key": series_row.canonical_brand_key,
            "source_brand_label": series_row.source_brand_label,
            "canonical_sector": series_row.canonical_sector,
            "canonical_category": series_row.canonical_category,
            "canonical_subcategory": series_row.canonical_subcategory,
            "source_subcategory_id": series_row.source_subcategory_id,
            "metric": series_row.metric,
            "unit": series_row.unit,
            "geography": series_row.geography,
            "methodology_cluster": series_row.methodology_cluster,
            "methodology_status": series_row.methodology_status,
            "analysis_period_count": len(analysis_periods),
            "analysis_periods": ";".join(map(str, analysis_periods)),
            "first_tbi": float(values[0]),
            "last_tbi": float(values[-1]),
            "mean_absolute_period_change": float(
                np.mean(np.abs(changes))
            ),
            "maximum_absolute_period_change": float(
                np.max(np.abs(changes))
            ),
            "tbi_range": float(np.max(values) - np.min(values)),
            "tbi_standard_deviation": float(np.std(values, ddof=0)),
            "methodology_caveat": series_row.methodology_caveat,
            "consistency_semantics": (
                "Lower mean absolute period change indicates greater "
                "observed stability only within the same comparable "
                "category-period-methodology block."
            ),
        }
    )

longitudinal_consistency_results = pd.DataFrame(consistency_rows)

CONSISTENCY_COMPARISON_KEYS = [
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "source_subcategory_id",
    "methodology_cluster",
    "analysis_periods",
]

longitudinal_consistency_results["comparison_series_count"] = (
    longitudinal_consistency_results.groupby(
        CONSISTENCY_COMPARISON_KEYS,
        dropna=False,
    )["source_label_series_key"].transform("count").astype(int)
)
longitudinal_consistency_results[
    "within_category_consistency_position"
] = (
    longitudinal_consistency_results.groupby(
        CONSISTENCY_COMPARISON_KEYS,
        dropna=False,
    )["mean_absolute_period_change"]
    .rank(method="min", ascending=True)
    .astype(int)
)

longitudinal_consistency_results = (
    longitudinal_consistency_results.sort_values(
        CONSISTENCY_COMPARISON_KEYS
        + [
            "within_category_consistency_position",
            "canonical_group",
            "source_brand_label",
        ]
    ).reset_index(drop=True)
)

print(
    longitudinal_consistency_results[
        [
            "canonical_group",
            "source_brand_label",
            "canonical_subcategory",
            "analysis_periods",
            "mean_absolute_period_change",
            "within_category_consistency_position",
            "comparison_series_count",
        ]
    ].to_string(index=False)
)


   canonical_group source_brand_label       canonical_subcategory    analysis_periods  mean_absolute_period_change  within_category_consistency_position  comparison_series_count
       Wings Group   TOP White Coffee                White Coffee 2022;2023;2024;2025                     3.033333                                     1                        1
       Wings Group          Floridina Ready-to-Drink Fruit Drinks 2022;2023;2024;2025                     1.533333                                     1                        3
       Wings Group            ale-ale Ready-to-Drink Fruit Drinks 2022;2023;2024;2025                     1.900000                                     2                        3
Unilever Indonesia            Buavita Ready-to-Drink Fruit Drinks 2022;2023;2024;2025                     2.733333                                     3                        3
          Indofood             Sarimi      Bagged Instant Noodles 2022;2023;2024;2025                     0.20

## Momentum Within Eligible Historical Series

Calculate net TBI change, average interval change, and interval direction only within Stage 3 momentum-eligible consecutive source-label series. No ordinary movement is calculated across the 2025–2026 methodology boundary.


In [10]:
momentum_eligible = longitudinal[
    longitudinal["momentum_eligibility"].eq("eligible_with_caveat")
].copy()

momentum_rows = []
for series_row in momentum_eligible.itertuples(index=False):
    series_key = series_row.source_label_series_key

    observations = competitive[
        competitive["source_label_series_key"].eq(series_key)
        & competitive["primary_analysis_eligible"].eq("yes")
        & competitive["analysis_scope"].eq("primary_strict_control")
        & competitive["value_available"].eq("yes")
        & competitive["metric"].eq("TBI")
        & competitive["unit"].eq("percent_index_points")
    ].copy()

    observations["reference_period_int"] = pd.to_numeric(
        observations["reference_period"],
        errors="raise",
    ).astype(int)
    observations["tbi_value"] = pd.to_numeric(
        observations["value"],
        errors="raise",
    )

    if observations["reference_period_int"].duplicated().any():
        raise ValueError(
            f"Duplicate periods found in momentum series {series_key}."
        )

    analysis_periods = longest_consecutive_block(
        observations["reference_period_int"].tolist()
    )
    if len(analysis_periods) < 3:
        raise ValueError(
            f"Eligible momentum series lacks a 3-period block: {series_key}"
        )

    block = (
        observations[
            observations["reference_period_int"].isin(analysis_periods)
        ]
        .sort_values("reference_period_int")
        .copy()
    )
    values = block["tbi_value"].to_numpy(dtype=float)
    changes = np.diff(values)
    net_change = float(values[-1] - values[0])

    tolerance = 1e-12
    if net_change > tolerance:
        momentum_direction = "positive"
    elif net_change < -tolerance:
        momentum_direction = "negative"
    else:
        momentum_direction = "flat"

    momentum_rows.append(
        {
            "source_label_series_key": series_key,
            "canonical_group": series_row.canonical_group,
            "canonical_brand_family": series_row.canonical_brand_family,
            "canonical_brand_key": series_row.canonical_brand_key,
            "source_brand_label": series_row.source_brand_label,
            "canonical_sector": series_row.canonical_sector,
            "canonical_category": series_row.canonical_category,
            "canonical_subcategory": series_row.canonical_subcategory,
            "source_subcategory_id": series_row.source_subcategory_id,
            "metric": series_row.metric,
            "unit": series_row.unit,
            "geography": series_row.geography,
            "methodology_cluster": series_row.methodology_cluster,
            "methodology_status": series_row.methodology_status,
            "analysis_period_count": len(analysis_periods),
            "analysis_periods": ";".join(map(str, analysis_periods)),
            "first_tbi": float(values[0]),
            "last_tbi": float(values[-1]),
            "net_tbi_change": net_change,
            "average_interval_tbi_change": float(np.mean(changes)),
            "positive_interval_count": int((changes > tolerance).sum()),
            "negative_interval_count": int((changes < -tolerance).sum()),
            "flat_interval_count": int(
                (np.abs(changes) <= tolerance).sum()
            ),
            "momentum_direction": momentum_direction,
            "methodology_caveat": series_row.methodology_caveat,
            "momentum_semantics": (
                "Net change is comparable only within the same "
                "category-period-methodology block; it is not a "
                "cross-category score."
            ),
        }
    )

momentum_results = pd.DataFrame(momentum_rows)

MOMENTUM_COMPARISON_KEYS = [
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "source_subcategory_id",
    "methodology_cluster",
    "analysis_periods",
]

momentum_results["comparison_series_count"] = (
    momentum_results.groupby(
        MOMENTUM_COMPARISON_KEYS,
        dropna=False,
    )["source_label_series_key"].transform("count").astype(int)
)
momentum_results["within_category_momentum_position"] = (
    momentum_results.groupby(
        MOMENTUM_COMPARISON_KEYS,
        dropna=False,
    )["net_tbi_change"]
    .rank(method="min", ascending=False)
    .astype(int)
)

momentum_results = (
    momentum_results.sort_values(
        MOMENTUM_COMPARISON_KEYS
        + [
            "within_category_momentum_position",
            "canonical_group",
            "source_brand_label",
        ]
    ).reset_index(drop=True)
)

print(
    momentum_results[
        [
            "canonical_group",
            "source_brand_label",
            "canonical_subcategory",
            "analysis_periods",
            "net_tbi_change",
            "momentum_direction",
            "within_category_momentum_position",
            "comparison_series_count",
        ]
    ].to_string(index=False)
)


   canonical_group source_brand_label       canonical_subcategory    analysis_periods  net_tbi_change momentum_direction  within_category_momentum_position  comparison_series_count
       Wings Group   TOP White Coffee                White Coffee 2022;2023;2024;2025             5.5           positive                                  1                        1
Unilever Indonesia            Buavita Ready-to-Drink Fruit Drinks 2022;2023;2024;2025             8.2           positive                                  1                        3
       Wings Group          Floridina Ready-to-Drink Fruit Drinks 2022;2023;2024;2025             1.8           positive                                  2                        3
       Wings Group            ale-ale Ready-to-Drink Fruit Drinks 2022;2023;2024;2025             0.1           positive                                  3                        3
          Indofood             Sarimi      Bagged Instant Noodles 2022;2023;2024;2025          

## Competitive Persistence

Evaluate only Stage 3 persistence-eligible historical category-methodology records. Track whether the baseline incumbent group remains a focal-group leader, whether the exact incumbent brand stays at or above the best other focal-group challenger, and how the incumbent-versus-challenger TBI gap changes across complete follow-up periods.


In [11]:
persistence_eligible = persistence[
    persistence["persistence_analysis_eligibility"].eq(
        "eligible_with_caveat"
    )
].copy()

persistence_rows = []


def filter_by_candidate(frame, candidate):
    # category_strength_results is already restricted to Top Brand,
    # so source_family does not need to be filtered again here.
    mask = (
        frame["source_category"].eq(candidate.source_category)
        & frame["source_subcategory"].eq(candidate.source_subcategory)
        & frame["source_subcategory_id"].eq(
            candidate.source_subcategory_id
        )
        & frame["canonical_sector"].eq(candidate.canonical_sector)
        & frame["canonical_category"].eq(candidate.canonical_category)
        & frame["canonical_subcategory"].eq(
            candidate.canonical_subcategory
        )
        & frame["methodology_cluster"].eq(
            candidate.methodology_cluster
        )
    )
    return frame.loc[mask].copy()


def group_leader_snapshot(period_frame):
    if period_frame.empty:
        raise ValueError("Persistence period frame is empty.")

    group_max = (
        period_frame.groupby("canonical_group", as_index=False)
        .agg(group_best_tbi=("tbi_value", "max"))
    )

    top_value = float(group_max["group_best_tbi"].max())

    leader_groups = sorted(
        group_max.loc[
            np.isclose(
                group_max["group_best_tbi"],
                top_value,
                rtol=0,
                atol=1e-12,
            ),
            "canonical_group",
        ].tolist()
    )

    leader_brand_parts = []

    for group in leader_groups:
        group_rows = period_frame[
            period_frame["canonical_group"].eq(group)
        ].copy()

        group_top = float(group_rows["tbi_value"].max())

        brand_rows = group_rows[
            np.isclose(
                group_rows["tbi_value"],
                group_top,
                rtol=0,
                atol=1e-12,
            )
        ]

        labels = " | ".join(
            sorted(brand_rows["source_brand_label"].unique())
        )

        leader_brand_parts.append(
            f"{group}: {labels}"
        )

    return (
        leader_groups,
        top_value,
        "; ".join(leader_brand_parts),
    )


for candidate in persistence_eligible.itertuples(index=False):
    category_frame = filter_by_candidate(
        category_strength_results,
        candidate,
    )

    baseline_period = int(candidate.baseline_period)

    follow_up_periods = [
        int(value)
        for value in candidate.complete_follow_up_periods.split(";")
        if value != ""
    ]

    if len(follow_up_periods) < 2:
        raise ValueError(
            "Persistence-eligible record has fewer than two "
            "follow-up periods."
        )

    all_required_periods = [
        baseline_period,
        *follow_up_periods,
    ]

    observed_periods = sorted(
        category_frame["reference_period"].astype(int).unique()
    )

    missing_periods = sorted(
        set(all_required_periods)
        - set(observed_periods)
    )

    if missing_periods:
        raise ValueError(
            "Persistence record is missing required periods: "
            f"{missing_periods}"
        )

    baseline_frame = category_frame[
        category_frame["reference_period"]
        .astype(int)
        .eq(baseline_period)
    ].copy()

    incumbent_baseline_rows = baseline_frame[
        baseline_frame["canonical_group"].eq(
            candidate.baseline_incumbent_group
        )
        & baseline_frame["canonical_brand_family"].eq(
            candidate.baseline_incumbent_brand
        )
    ]

    if incumbent_baseline_rows.empty:
        raise ValueError(
            "Baseline incumbent brand is absent from the "
            "baseline period."
        )

    incumbent_baseline_tbi = float(
        incumbent_baseline_rows["tbi_value"].max()
    )

    best_other_baseline_rows = baseline_frame[
        ~baseline_frame["canonical_group"].eq(
            candidate.baseline_incumbent_group
        )
    ]

    if best_other_baseline_rows.empty:
        raise ValueError(
            "No other focal-group challenger exists in the "
            "baseline period."
        )

    best_other_baseline = float(
        best_other_baseline_rows["tbi_value"].max()
    )

    baseline_gap = (
        incumbent_baseline_tbi
        - best_other_baseline
    )

    followup_snapshots = []
    followup_gaps = []
    group_led_flags = []
    incumbent_not_below_flags = []

    for period in follow_up_periods:
        period_frame = category_frame[
            category_frame["reference_period"]
            .astype(int)
            .eq(period)
        ].copy()

        (
            leader_groups,
            leader_tbi,
            leader_brands,
        ) = group_leader_snapshot(
            period_frame
        )

        incumbent_rows = period_frame[
            period_frame["canonical_group"].eq(
                candidate.baseline_incumbent_group
            )
            & period_frame["canonical_brand_family"].eq(
                candidate.baseline_incumbent_brand
            )
        ]

        if incumbent_rows.empty:
            raise ValueError(
                "Incumbent brand absent from follow-up period "
                f"{period}."
            )

        incumbent_tbi = float(
            incumbent_rows["tbi_value"].max()
        )

        other_rows = period_frame[
            ~period_frame["canonical_group"].eq(
                candidate.baseline_incumbent_group
            )
        ]

        if other_rows.empty:
            raise ValueError(
                "No other focal-group challenger in follow-up "
                f"period {period}."
            )

        best_other_tbi = float(
            other_rows["tbi_value"].max()
        )

        gap = incumbent_tbi - best_other_tbi

        group_led = (
            candidate.baseline_incumbent_group
            in leader_groups
        )

        incumbent_not_below = gap >= -1e-12

        group_led_flags.append(group_led)
        incumbent_not_below_flags.append(
            incumbent_not_below
        )
        followup_gaps.append(gap)

        followup_snapshots.append(
            {
                "period": period,
                "leader_groups": leader_groups,
                "leader_tbi": leader_tbi,
                "leader_brands": leader_brands,
                "incumbent_tbi": incumbent_tbi,
                "best_other_tbi": best_other_tbi,
                "gap": gap,
            }
        )

    first_group_lead_loss_period = ""

    for snapshot, flag in zip(
        followup_snapshots,
        group_led_flags,
    ):
        if not flag:
            first_group_lead_loss_period = (
                snapshot["period"]
            )
            break

    first_brand_overtake_period = ""

    for snapshot, flag in zip(
        followup_snapshots,
        incumbent_not_below_flags,
    ):
        if not flag:
            first_brand_overtake_period = (
                snapshot["period"]
            )
            break

    final_snapshot = followup_snapshots[-1]
    final_gap = float(followup_gaps[-1])

    persistence_rows.append(
        {
            "source_family": (
                candidate.source_family
            ),
            "source_category": (
                candidate.source_category
            ),
            "source_subcategory": (
                candidate.source_subcategory
            ),
            "source_subcategory_id": (
                candidate.source_subcategory_id
            ),
            "canonical_sector": (
                candidate.canonical_sector
            ),
            "canonical_category": (
                candidate.canonical_category
            ),
            "canonical_subcategory": (
                candidate.canonical_subcategory
            ),
            "methodology_cluster": (
                candidate.methodology_cluster
            ),
            "baseline_period": baseline_period,
            "baseline_incumbent_group": (
                candidate.baseline_incumbent_group
            ),
            "baseline_incumbent_brand": (
                candidate.baseline_incumbent_brand
            ),
            "baseline_incumbent_observation_id": (
                candidate.baseline_incumbent_observation_id
            ),
            "baseline_incumbent_tbi": (
                incumbent_baseline_tbi
            ),
            "baseline_gap_to_best_other_focal_group": (
                float(baseline_gap)
            ),
            "follow_up_period_count": (
                len(follow_up_periods)
            ),
            "follow_up_periods": ";".join(
                map(str, follow_up_periods)
            ),
            "incumbent_group_led_follow_up_count": (
                int(sum(group_led_flags))
            ),
            "incumbent_group_led_all_follow_ups": (
                "yes"
                if all(group_led_flags)
                else "no"
            ),
            (
                "incumbent_brand_not_below_best_other_"
                "group_all_follow_ups"
            ): (
                "yes"
                if all(
                    incumbent_not_below_flags
                )
                else "no"
            ),
            "first_group_lead_loss_period": (
                first_group_lead_loss_period
            ),
            "first_brand_overtake_period": (
                first_brand_overtake_period
            ),
            "overtaken_by_other_focal_group": (
                "yes"
                if any(
                    not flag
                    for flag
                    in incumbent_not_below_flags
                )
                else "no"
            ),
            "final_follow_up_period": (
                final_snapshot["period"]
            ),
            "final_leader_groups": " | ".join(
                final_snapshot["leader_groups"]
            ),
            "final_leader_brand_labels": (
                final_snapshot["leader_brands"]
            ),
            "final_incumbent_tbi": float(
                final_snapshot["incumbent_tbi"]
            ),
            "final_best_other_focal_group_tbi": (
                float(
                    final_snapshot[
                        "best_other_tbi"
                    ]
                )
            ),
            "final_gap_to_best_other_focal_group": (
                final_gap
            ),
            "incumbent_gap_change_from_baseline": (
                float(
                    final_gap
                    - baseline_gap
                )
            ),
            "follow_up_leadership_trace": "; ".join(
                (
                    f"{snapshot['period']}:"
                    f"{' | '.join(snapshot['leader_groups'])}"
                )
                for snapshot
                in followup_snapshots
            ),
            "persistence_semantics": (
                "Persistence is evaluated only against "
                "complete focal-group follow-up periods "
                "within one historical methodology cluster."
            ),
            "methodology_caveat": (
                "Historical Top Brand methodology is not "
                "independently verified."
            ),
        }
    )

competitive_persistence_results = (
    pd.DataFrame(persistence_rows)
    .sort_values(
        [
            "source_subcategory_id",
            "methodology_cluster",
        ]
    )
    .reset_index(drop=True)
)

print(
    competitive_persistence_results[
        [
            "canonical_subcategory",
            "baseline_period",
            "baseline_incumbent_group",
            "baseline_incumbent_brand",
            "follow_up_periods",
            "incumbent_group_led_all_follow_ups",
            "overtaken_by_other_focal_group",
            "final_leader_groups",
            "incumbent_gap_change_from_baseline",
        ]
    ].to_string(index=False)
)

      canonical_subcategory  baseline_period baseline_incumbent_group baseline_incumbent_brand follow_up_periods incumbent_group_led_all_follow_ups overtaken_by_other_focal_group final_leader_groups  incumbent_gap_change_from_baseline
Ready-to-Drink Fruit Drinks             2022       Unilever Indonesia                  Buavita    2023;2024;2025                                yes                             no  Unilever Indonesia                                 6.4
       Antiseptic Bath Soap             2022       Unilever Indonesia                 Lifebuoy    2023;2024;2025                                yes                             no  Unilever Indonesia                                 1.3
        Cup Instant Noodles             2022                 Indofood                  Pop Mie    2023;2024;2025                                 no                            yes         Wings Group                               -51.9
            Sweet Soy Sauce             2022       Unilever 

## Dimension-Level Execution Summary

Reconcile the calculated and non-computable dimensions against the pre-specified Stage 3 eligibility rules. Consumer reach, structural category breadth, portfolio concentration, and overall portfolio leadership remain uncalculated or deferred exactly as required.


In [12]:
structural_leaders = structural_brand_breadth[
    structural_brand_breadth["breadth_rank"].eq(1)
]
structural_leader_text = " | ".join(
    (
        f"{row.canonical_group}="
        f"{row.structural_brand_family_count}"
    )
    for row in structural_leaders.sort_values(
        "canonical_group"
    ).itertuples()
)

leader_event_counts = {group: 0 for group in FOCAL_GROUPS}
leader_category_sets = {group: set() for group in FOCAL_GROUPS}

for row in category_leadership_results.itertuples(index=False):
    groups = [
        value.strip()
        for value in str(row.leader_groups).split("|")
        if value.strip()
    ]
    for group in groups:
        if group in leader_event_counts:
            leader_event_counts[group] += 1
            leader_category_sets[group].add(row.canonical_subcategory)

leadership_event_text = "; ".join(
    (
        f"{group}: leader-period events={leader_event_counts[group]}, "
        f"distinct categories led={len(leader_category_sets[group])}"
    )
    for group in FOCAL_GROUPS
)

execution_status_map = {
    "DIM01A": "calculated_with_caveat",
    "DIM01B": "not_computed_not_eligible",
    "DIM02": "calculated_with_caveat",
    "DIM03": "not_computed_not_eligible",
    "DIM04": "calculated_with_caveat",
    "DIM05": "calculated_conditionally",
    "DIM06": "calculated_conditionally",
    "DIM07": "calculated_conditionally",
    "DIM08": "not_computed_not_eligible",
    "DIM09": "calculated_conditionally",
    "DIM10": "deferred",
}

result_file_map = {
    "DIM01A": "data/analytical/structural_brand_breadth_results.csv",
    "DIM01B": "",
    "DIM02": "data/analytical/competitive_breadth_results.csv",
    "DIM03": "",
    "DIM04": "data/analytical/category_strength_results.csv",
    "DIM05": "data/analytical/category_leadership_results.csv",
    "DIM06": "data/analytical/longitudinal_consistency_results.csv",
    "DIM07": "data/analytical/competitive_persistence_results.csv",
    "DIM08": "",
    "DIM09": "data/analytical/momentum_results.csv",
    "DIM10": "",
}

row_count_map = {
    "DIM01A": len(structural_brand_breadth),
    "DIM01B": 0,
    "DIM02": len(competitive_breadth),
    "DIM03": 0,
    "DIM04": len(category_strength_results),
    "DIM05": len(category_leadership_results),
    "DIM06": len(longitudinal_consistency_results),
    "DIM07": len(competitive_persistence_results),
    "DIM08": 0,
    "DIM09": len(momentum_results),
    "DIM10": 0,
}

descriptive_result_map = {
    "DIM01A": (
        "Structural brand-family breadth leader(s): "
        f"{structural_leader_text}"
    ),
    "DIM01B": (
        "Not calculated because comparable structural category mapping "
        "remains incomplete."
    ),
    "DIM02": (
        "Denominator-explicit counts reported without performance ranking "
        "because public-source coverage remains selective."
    ),
    "DIM03": (
        "No comparable CRP universe is available across focal groups; "
        "limited Brand Footprint facts remain context only."
    ),
    "DIM04": (
        "Observed TBI strength is retained at brand-category-period-"
        "methodology level only; no cross-category TBI average is created."
    ),
    "DIM05": leadership_event_text,
    "DIM06": (
        "Consistency is calculated per eligible source-label series and "
        "positioned only within directly comparable category-period blocks."
    ),
    "DIM07": (
        f"{len(competitive_persistence_results)} eligible historical "
        "category-methodology persistence records evaluated."
    ),
    "DIM08": (
        "Not calculated because the current performance evidence is not "
        "an additive, sufficiently complete portfolio distribution."
    ),
    "DIM09": (
        "Momentum is calculated per eligible source-label series and "
        "positioned only within directly comparable category-period blocks."
    ),
    "DIM10": (
        "Overall portfolio leadership remains deferred; no composite score "
        "or overall winner is calculated in Stage 4A."
    ),
}

stage4_caveat_map = {
    "DIM01A": (
        "Brand-family breadth is countable, while structural category "
        "mapping remains incomplete."
    ),
    "DIM01B": (
        "Wings Group structural category mapping is not sufficiently "
        "complete for comparable category breadth."
    ),
    "DIM02": (
        "Selective public coverage is a structural limitation and must not "
        "be interpreted as performance."
    ),
    "DIM03": (
        "Rank, lower bounds, qualitative facts, and public-summary omission "
        "are not CRP."
    ),
    "DIM04": (
        "TBI is source-native and is not market share; comparisons stay "
        "within one category-period-methodology cluster."
    ),
    "DIM05": (
        "Leadership is focal-group leadership only and requires complete "
        "pre-specified focal denominators."
    ),
    "DIM06": (
        "Historical Top Brand methodology is not independently verified; "
        "2026 is not joined to 2022–2025."
    ),
    "DIM07": (
        "Persistence is descriptive and does not establish a causal "
        "competitive mechanism."
    ),
    "DIM08": (
        "HHI is prohibited for the current non-additive cross-category "
        "metric universe."
    ),
    "DIM09": (
        "Historical Top Brand methodology is not independently verified; "
        "2025–2026 movement is not calculated."
    ),
    "DIM10": (
        "Dimension-level evidence must be validated and assessed for "
        "compatibility before any synthesis."
    ),
}

dimension_level_summary = metric_rules.copy()
dimension_level_summary["stage4a_execution_status"] = (
    dimension_level_summary["dimension_id"].map(execution_status_map)
)
dimension_level_summary["result_file"] = (
    dimension_level_summary["dimension_id"].map(result_file_map)
)
dimension_level_summary["result_row_count"] = (
    dimension_level_summary["dimension_id"].map(row_count_map).astype(int)
)
dimension_level_summary["descriptive_result"] = (
    dimension_level_summary["dimension_id"].map(descriptive_result_map)
)
dimension_level_summary["stage4a_caveat"] = (
    dimension_level_summary["dimension_id"].map(stage4_caveat_map)
)

if dimension_level_summary[
    [
        "stage4a_execution_status",
        "result_file",
        "descriptive_result",
        "stage4a_caveat",
    ]
].isna().any().any():
    raise ValueError(
        "At least one Stage 3 dimension rule lacks a Stage 4A treatment."
    )

print(
    dimension_level_summary[
        [
            "dimension_id",
            "dimension",
            "current_eligibility",
            "stage4a_execution_status",
            "result_row_count",
            "descriptive_result",
        ]
    ].to_string(index=False)
)


dimension_id                    dimension    current_eligibility  stage4a_execution_status  result_row_count                                                                                                                                                                                                                                             descriptive_result
      DIM01A     structural_brand_breadth   eligible_with_caveat    calculated_with_caveat                 4                                                                                                                                                                                                         Structural brand-family breadth leader(s): Indofood=38
      DIM01B  structural_category_breadth not_eligible_currently not_computed_not_eligible                 0                                                                                                                                                                    

## Stage 4A Validation

Validate input provenance, ownership isolation, denominator use, source-native TBI semantics, longitudinal boundaries, persistence eligibility, prohibited calculations, and output completeness before writing any analytical result.


In [13]:
validation_rows = []

def add_check(
    check_id,
    validation_area,
    description,
    condition,
    result,
    required_treatment,
    notes="",
    caveat=False,
    critical=True,
):
    condition = bool(condition)
    status = (
        "passed_with_caveat"
        if condition and caveat
        else "passed"
        if condition
        else "failed"
    )
    validation_rows.append(
        {
            "check_id": check_id,
            "validation_area": validation_area,
            "check_description": description,
            "result": str(result),
            "status": status,
            "critical_failure": (
                "yes" if (not condition and critical) else "no"
            ),
            "required_treatment": required_treatment,
            "notes": notes,
        }
    )


add_check(
    "S4A001",
    "input_manifest",
    "The Stage 3 output manifest matches the locked expected SHA-256.",
    actual_manifest_sha256 == EXPECTED_MANIFEST_SHA256,
    "manifest checksum matched",
    "Stop analysis if the Stage 3 manifest changes.",
)

add_check(
    "S4A002",
    "input_integrity",
    "All manifest-tracked Stage 3 inputs match committed row counts and SHA-256 values.",
    (
        input_integrity["row_count_status"].eq("passed").all()
        and input_integrity["sha256_status"].eq("passed").all()
    ),
    f"{len(input_integrity)}/{len(input_integrity)} tracked inputs passed",
    "Stop analysis if any tracked input differs.",
)

add_check(
    "S4A003",
    "prior_stage_gate",
    "Stage 3 contains no critical failure and retains PASS_WITH_CAVEAT.",
    (
        not stage3_validation["critical_failure"].str.lower().eq("yes").any()
        and stage3_gate_rows.iloc[0]["result"] == "PASS_WITH_CAVEAT"
    ),
    "PASS_WITH_CAVEAT",
    "Do not proceed from a failed prior stage.",
    caveat=True,
)

expected_structural_counts = (
    structural_primary.groupby("group")["structural_brand_count_key"]
    .nunique()
    .reindex(FOCAL_GROUPS, fill_value=0)
)
actual_structural_counts = (
    structural_brand_breadth.set_index("canonical_group")[
        "structural_brand_family_count"
    ]
    .reindex(FOCAL_GROUPS, fill_value=0)
)

add_check(
    "S4A004",
    "structural_brand_breadth",
    "Structural brand-family counts exactly reconcile to the current strict-control primary ownership universe.",
    expected_structural_counts.equals(actual_structural_counts),
    f"{len(structural_brand_breadth)} focal-group breadth rows",
    "Count only distinct current strict-control canonical brand-family keys.",
    caveat=True,
    notes="Structural category breadth remains separately ineligible.",
)

add_check(
    "S4A005",
    "structural_category_breadth",
    "Structural category breadth remains uncalculated.",
    (
        dimension_level_summary.loc[
            dimension_level_summary["dimension_id"].eq("DIM01B"),
            "stage4a_execution_status",
        ].eq("not_computed_not_eligible").all()
        and structural_brand_breadth[
            "structural_category_breadth_status"
        ].eq("not_eligible_currently").all()
    ),
    "not_eligible_currently",
    "Do not infer ownership categories from survey visibility.",
    caveat=True,
)

expanded_target_member_count = int(
    (
        denominator_numeric["eligible_target_count"]
        + denominator_numeric["pending_ownership_target_count"]
        + denominator_numeric["sensitivity_target_count"]
    ).sum()
)

add_check(
    "S4A006",
    "competitive_denominators",
    "Expanded target-period members reconcile to the Stage 3 denominator universe.",
    expanded_target_member_count == 104,
    f"{expanded_target_member_count} target-period members",
    "Keep eligible, pending-ownership, and sensitivity members separate.",
)

add_check(
    "S4A007",
    "competitive_breadth",
    "Competitive breadth reports explicit denominator components and does not create a performance rank.",
    (
        set(competitive_breadth["canonical_group"]).issubset(
            set(FOCAL_GROUPS)
        )
        and "performance_rank" not in competitive_breadth.columns
        and competitive_breadth[
            "performance_ranking_status"
        ].eq("not_ranked_due_selective_public_coverage").all()
    ),
    f"{len(competitive_breadth)} group-source rows",
    "Treat selective coverage as a limitation, not a performance outcome.",
    caveat=True,
)

add_check(
    "S4A008",
    "metric_semantics",
    "Category-strength results contain only source-native TBI values in percent index points.",
    (
        category_strength_results["metric"].eq("TBI").all()
        and category_strength_results["unit"].eq(
            "percent_index_points"
        ).all()
    ),
    f"{len(category_strength_results)} TBI observations",
    "Do not relabel TBI as market share.",
)

strength_source_ids = set(
    competitive.loc[strength_mask, "observation_id"]
)
strength_output_ids = set(
    category_strength_results["observation_id"]
)

add_check(
    "S4A009",
    "category_strength_scope",
    "Every eligible category-strength candidate is retained exactly once.",
    (
        strength_source_ids == strength_output_ids
        and category_strength_results["observation_id"].is_unique
    ),
    f"{len(strength_output_ids)} unique observations",
    "Use only observed strict-control primary TBI candidates.",
)

add_check(
    "S4A010",
    "ownership_isolation",
    "Ownership-pending Indocafe observations are excluded from primary performance calculations.",
    (
        not category_strength_results[
            "canonical_brand_family"
        ].eq("Indocafe").any()
        and not longitudinal_consistency_results[
            "canonical_brand_family"
        ].eq("Indocafe").any()
        and not momentum_results[
            "canonical_brand_family"
        ].eq("Indocafe").any()
    ),
    "Indocafe excluded from primary calculations",
    "Add authoritative primary ownership support before primary use.",
    caveat=True,
)

add_check(
    "S4A011",
    "sensitivity_isolation",
    "Le Minerale remains excluded from strict-control primary results.",
    not category_strength_results[
        "canonical_brand_family"
    ].eq("Le Minerale").any(),
    "Le Minerale absent from primary performance outputs",
    "Keep Le Minerale in extended-group sensitivity evidence only.",
)

leadership_valid = True
if not category_leadership_results.empty:
    leadership_valid = (
        category_leadership_results[
            "category_denominator_status"
        ].eq("complete_observed").all()
        and category_leadership_results["focal_group_count"].ge(2).all()
        and category_leadership_results["result_scope"].eq(
            "focal-group leader only; not full-market category leader"
        ).all()
    )

add_check(
    "S4A012",
    "category_leadership",
    "Category leadership is produced only from complete multi-group focal denominators.",
    leadership_valid,
    f"{len(category_leadership_results)} eligible category-period results",
    "Do not call a focal-group leader the full-market category leader.",
)

add_check(
    "S4A013",
    "longitudinal_eligibility",
    "Consistency results contain exactly the Stage 3 eligible source-label series.",
    (
        set(
            longitudinal_consistency_results[
                "source_label_series_key"
            ]
        )
        == set(
            consistency_eligible["source_label_series_key"]
        )
        and len(longitudinal_consistency_results) == 14
    ),
    f"{len(longitudinal_consistency_results)} eligible series",
    "Calculate consistency only for screened series.",
    caveat=True,
)

add_check(
    "S4A014",
    "longitudinal_boundary",
    "Consistency calculations use at least three consecutive periods and never bridge into 2026.",
    (
        longitudinal_consistency_results[
            "analysis_period_count"
        ].ge(3).all()
        and ~longitudinal_consistency_results[
            "analysis_periods"
        ].str.contains("2026", regex=False).any()
    ),
    "No 2025–2026 bridge",
    "Keep historical and current methodology clusters separate.",
)

add_check(
    "S4A015",
    "consistency_comparability",
    "Consistency positions are defined only within the same category, methodology cluster, and analyzed period block.",
    "within_category_consistency_position"
    in longitudinal_consistency_results.columns,
    "within-category positions only",
    "Do not create a cross-category consistency score.",
)

add_check(
    "S4A016",
    "momentum_eligibility",
    "Momentum results contain exactly the Stage 3 eligible source-label series.",
    (
        set(momentum_results["source_label_series_key"])
        == set(momentum_eligible["source_label_series_key"])
        and len(momentum_results) == 14
    ),
    f"{len(momentum_results)} eligible series",
    "Calculate momentum only for screened source-label series.",
    caveat=True,
)

add_check(
    "S4A017",
    "momentum_boundary",
    "Momentum calculations never treat 2025–2026 as an ordinary interval.",
    ~momentum_results[
        "analysis_periods"
    ].str.contains("2026", regex=False).any(),
    "No 2025–2026 movement calculated",
    "Keep methodology-status boundaries intact.",
)

add_check(
    "S4A018",
    "momentum_comparability",
    "Momentum positions are defined only within the same category, methodology cluster, and analyzed period block.",
    "within_category_momentum_position" in momentum_results.columns,
    "within-category positions only",
    "Do not create a cross-category momentum score.",
)

add_check(
    "S4A019",
    "persistence_eligibility",
    "Competitive persistence is calculated only for the four Stage 3 eligible category-methodology records.",
    (
        len(competitive_persistence_results) == 4
        and len(persistence_eligible) == 4
        and competitive_persistence_results[
            "methodology_cluster"
        ].eq("top_brand_historical_unverified").all()
    ),
    f"{len(competitive_persistence_results)} persistence records",
    "Require a complete baseline and at least two complete follow-up periods.",
    caveat=True,
)

add_check(
    "S4A020",
    "persistence_follow_up",
    "Every persistence result contains at least two complete follow-up periods.",
    competitive_persistence_results[
        "follow_up_period_count"
    ].ge(2).all(),
    "All persistence records have at least two follow-up periods",
    "Do not calculate persistence from incomplete follow-up denominators.",
)

add_check(
    "S4A021",
    "consumer_reach",
    "Consumer reach remains uncalculated for cross-group quantitative comparison.",
    dimension_level_summary.loc[
        dimension_level_summary["dimension_id"].eq("DIM03"),
        "stage4a_execution_status",
    ].eq("not_computed_not_eligible").all(),
    "not_eligible_currently",
    "Acquire a comparable source-native CRP universe before calculation.",
    caveat=True,
)

add_check(
    "S4A022",
    "portfolio_concentration",
    "Performance-weighted HHI remains uncalculated.",
    dimension_level_summary.loc[
        dimension_level_summary["dimension_id"].eq("DIM08"),
        "stage4a_execution_status",
    ].eq("not_computed_not_eligible").all(),
    "HHI not calculated",
    "Do not calculate HHI from cross-category TBI or mixed metrics.",
)

add_check(
    "S4A023",
    "overall_leadership",
    "Overall portfolio leadership remains deferred.",
    dimension_level_summary.loc[
        dimension_level_summary["dimension_id"].eq("DIM10"),
        "stage4a_execution_status",
    ].eq("deferred").all(),
    "Overall winner deferred",
    "Complete validation and later synthesis before assessing defensibility.",
)

RESULT_DATAFRAMES = {
    "structural_brand_breadth_results": structural_brand_breadth,
    "competitive_breadth_results": competitive_breadth,
    "category_strength_results": category_strength_results,
    "category_leadership_results": category_leadership_results,
    "longitudinal_consistency_results": longitudinal_consistency_results,
    "momentum_results": momentum_results,
    "competitive_persistence_results": competitive_persistence_results,
    "dimension_level_summary": dimension_level_summary,
}

for name, frame in RESULT_DATAFRAMES.items():
    forbidden_columns = [
        column
        for column in frame.columns
        if column.lower() in {
            "composite_score",
            "overall_score",
            "overall_winner",
            "performance_hhi",
        }
    ]
    if forbidden_columns:
        raise ValueError(
            f"Forbidden synthesis columns found in {name}: "
            f"{forbidden_columns}"
        )

add_check(
    "S4A024",
    "prohibited_calculations",
    "No composite score, overall winner field, or performance HHI is produced.",
    True,
    "No prohibited synthesis fields",
    "Keep Stage 4A at dimension level.",
)

add_check(
    "S4A025",
    "output_scope",
    "All eight analytical result tables are available before publication.",
    len(RESULT_DATAFRAMES) == 8,
    f"{len(RESULT_DATAFRAMES)} analytical result tables",
    "Write only approved Stage 4A analytical outputs.",
)

stage4a_validation = pd.DataFrame(validation_rows)

if stage4a_validation["status"].eq("failed").any():
    final_gate = "FAIL"
else:
    final_gate = (
        "PASS_WITH_CAVEAT"
        if stage4a_validation["status"].eq(
            "passed_with_caveat"
        ).any()
        else "PASS"
    )

stage_gate_row = pd.DataFrame(
    [
        {
            "check_id": "S4A026",
            "validation_area": "stage_gate",
            "check_description": (
                "Stage 4A dimension-level analysis is complete enough "
                "for execution review."
            ),
            "result": final_gate,
            "status": (
                "passed_with_caveat"
                if final_gate == "PASS_WITH_CAVEAT"
                else "passed"
                if final_gate == "PASS"
                else "failed"
            ),
            "critical_failure": (
                "yes" if final_gate == "FAIL" else "no"
            ),
            "required_treatment": (
                "Review the executed notebook and generated outputs "
                "before repository publication."
            ),
            "notes": (
                "No composite score or overall portfolio winner is "
                "produced in Stage 4A."
            ),
        }
    ]
)
stage4a_validation = pd.concat(
    [stage4a_validation, stage_gate_row],
    ignore_index=True,
)

print(
    stage4a_validation[
        [
            "check_id",
            "validation_area",
            "result",
            "status",
        ]
    ].to_string(index=False)
)
print(f"Final gate: {final_gate}")

if final_gate == "FAIL":
    failed_checks = stage4a_validation.loc[
        stage4a_validation["status"].eq("failed"),
        "check_id",
    ].tolist()
    raise RuntimeError(
        f"Stage 4A validation failed: {failed_checks}"
    )


check_id             validation_area                                                      result             status
  S4A001              input_manifest                                   manifest checksum matched             passed
  S4A002             input_integrity                                   7/7 tracked inputs passed             passed
  S4A003            prior_stage_gate                                            PASS_WITH_CAVEAT passed_with_caveat
  S4A004    structural_brand_breadth                                  4 focal-group breadth rows passed_with_caveat
  S4A005 structural_category_breadth                                      not_eligible_currently passed_with_caveat
  S4A006    competitive_denominators                                   104 target-period members             passed
  S4A007         competitive_breadth                                         8 group-source rows passed_with_caveat
  S4A008            metric_semantics                                    

## Deterministic Output Writing and Manifest

Write the approved analytical and validation outputs with stable CSV line endings, calculate their SHA-256 values, and create the Stage 4A output manifest. No Git, commit, push, archive, or publishing operation is performed in this notebook.


In [14]:
OUTPUT_SPECS = [
    (
        "data/analytical/structural_brand_breadth_results.csv",
        structural_brand_breadth,
        (
            "Current strict-control structural brand-family breadth by "
            "focal group with category-mapping caveats"
        ),
    ),
    (
        "data/analytical/competitive_breadth_results.csv",
        competitive_breadth,
        (
            "Denominator-explicit competitive breadth counts by group "
            "and source family without coverage-as-performance ranking"
        ),
    ),
    (
        "data/analytical/category_strength_results.csv",
        category_strength_results,
        (
            "Observed strict-control Top Brand TBI strength within "
            "category-period-methodology clusters"
        ),
    ),
    (
        "data/analytical/category_leadership_results.csv",
        category_leadership_results,
        (
            "Conditional focal-group category leadership results from "
            "complete category-period denominators"
        ),
    ),
    (
        "data/analytical/longitudinal_consistency_results.csv",
        longitudinal_consistency_results,
        (
            "Source-label longitudinal consistency results for Stage 3 "
            "eligible historical series"
        ),
    ),
    (
        "data/analytical/momentum_results.csv",
        momentum_results,
        (
            "Within-series historical Top Brand momentum results without "
            "bridging the 2025-2026 methodology boundary"
        ),
    ),
    (
        "data/analytical/competitive_persistence_results.csv",
        competitive_persistence_results,
        (
            "Persistence outcomes for Stage 3 eligible complete "
            "category-methodology baselines and follow-up periods"
        ),
    ),
    (
        "data/analytical/dimension_level_summary.csv",
        dimension_level_summary,
        (
            "Stage 4A dimension-level execution summary preserving "
            "non-computable and deferred dimensions"
        ),
    ),
    (
        "metadata/stage4_analysis_validation.csv",
        stage4a_validation,
        "Stage 4A analytical validation and final gate",
    ),
]

for relative_path, frame, _ in OUTPUT_SPECS:
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        destination,
        index=False,
        lineterminator="\n",
    )

manifest_rows = []
for relative_path, frame, description in OUTPUT_SPECS:
    path = OUTPUT_ROOT / relative_path
    manifest_rows.append(
        {
            "file_path": relative_path,
            "row_count": len(frame),
            "sha256": sha256_file(path),
            "description": description,
            "repository_head": REPOSITORY_HEAD,
        }
    )

stage4_manifest = pd.DataFrame(manifest_rows)
manifest_output_path = METADATA_ROOT / "stage4_output_manifest.csv"
stage4_manifest.to_csv(
    manifest_output_path,
    index=False,
    lineterminator="\n",
)

written_paths = [
    str(OUTPUT_ROOT / relative_path)
    for relative_path, _, _ in OUTPUT_SPECS
] + [str(manifest_output_path)]

missing_written = [
    path for path in written_paths if not Path(path).exists()
]
if missing_written:
    raise FileNotFoundError(
        f"Stage 4A output write failed: {missing_written}"
    )

print(stage4_manifest.to_string(index=False))
print(f"Stage 4A final gate: {final_gate}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Output files written: {len(written_paths)}")


                                           file_path  row_count                                                           sha256                                                                                                        description                          repository_head
data/analytical/structural_brand_breadth_results.csv          4 edb6ae4f6e5c20f187498f1f0f78e0884b5ffa6b60a5e0aad5f807346377b954                Current strict-control structural brand-family breadth by focal group with category-mapping caveats 2648ddbe2b902f79826778a326e64ea9dc031e0f
     data/analytical/competitive_breadth_results.csv          8 077a734c33ca69ffe5ed7266a8d59dcbdba28e06b96ea85019cb3196829cfe3f Denominator-explicit competitive breadth counts by group and source family without coverage-as-performance ranking 2648ddbe2b902f79826778a326e64ea9dc031e0f
       data/analytical/category_strength_results.csv         75 5e9956b919baa324803e2e3b671f69799b326919b47156a4faf5a75826362019                 